# E3 YOLO11n COCO Pretrained — epoch 10 finalization

Read and package the verified epoch-10 result. This notebook does not train, validate on TEST, or alter ROD Final. The original training schedule was 30 epochs; the current project budget stops at epoch 10.


In [ ]:
SOURCE_ZIP = None  # optional exact /kaggle/input/.../results11.zip path
OUTPUT = '/kaggle/working/rod_e3_pretrained_epoch010.zip'
FROZEN_YAML = None  # optional exact path to e3_pretrained.yaml


In [ ]:
import importlib.metadata, subprocess, sys
try:
    version = importlib.metadata.version('ultralytics')
except importlib.metadata.PackageNotFoundError:
    version = None
if version != '8.4.165':
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'ultralytics==8.4.165'])
import torch, yaml
from ultralytics import YOLO
print('torch', torch.__version__, 'ultralytics', importlib.metadata.version('ultralytics'))


In [ ]:
from __future__ import annotations
import os
import re
import shutil
import tempfile
from datetime import datetime, timezone
def sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def canonical_hash(obj):
    return hashlib.sha256(json.dumps(obj, sort_keys=True, ensure_ascii=False,
                                     separators=(",", ":")).encode()).hexdigest()

def normalize_effective_arg(key, value):
    """Normalize values whose YAML/runtime representations are semantically equal."""
    if key == "device":
        if isinstance(value, (list, tuple)):
            return ",".join(str(item).strip() for item in value)
        if value is None:
            return None
        return ",".join(part.strip() for part in str(value).strip().split(","))
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, tuple):
        return [normalize_effective_arg(key, item) for item in value]
    return value

def effective_args_mismatches(expected, actual, ignored=None):
    ignored = set(ignored or ())
    mismatches = {}
    for key, expected_value in expected.items():
        if key in ignored:
            continue
        actual_value = actual.get(key)
        if normalize_effective_arg(key, actual_value) != normalize_effective_arg(key, expected_value):
            mismatches[key] = {"actual": actual_value, "expected": expected_value}
    return mismatches

def safe_extract(archive, destination):
    destination = Path(destination).resolve()
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive) as z:
        if z.testzip() is not None:
            raise ValueError("ZIP CRC fail")
        for item in z.infolist():
            name = item.filename.replace("\\", "/")
            if name.startswith("/") or ":" in name or not (destination / name).resolve().is_relative_to(destination):
                raise ValueError(f"Unsafe ZIP member: {item.filename}")
        z.extractall(destination)

def inspect_checkpoint(path, expected_epoch, torch_module):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(path)
    ck = torch_module.load(path, map_location="cpu", weights_only=False)
    if not isinstance(ck, dict) or ck.get("epoch") != expected_epoch - 1:
        raise ValueError(f"Wrong checkpoint epoch: {path}")
    if not isinstance(ck.get("optimizer"), dict) or not ck["optimizer"].get("state") or not ck["optimizer"].get("param_groups"):
        raise ValueError(f"Optimizer state missing: {path}")
    if ck.get("ema") is None or ck.get("scaler") is None:
        raise ValueError(f"EMA/scaler state missing: {path}")
    if ck.get("train_args", {}).get("epochs") != 30:
        raise ValueError(f"Checkpoint schedule is not 30 epochs: {path}")
    info = {"path": str(path), "epoch": expected_epoch, "raw_epoch": ck.get("epoch"),
            "bytes": path.stat().st_size,
            "sha256": sha256(path), "optimizer_state_entries": len(ck["optimizer"]["state"]),
            "optimizer_param_groups": len(ck["optimizer"]["param_groups"]),
            "ema_present": ck.get("ema") is not None, "ema_type": type(ck.get("ema")).__name__,
            "scaler_present": ck.get("scaler") is not None,
            "scaler_type": type(ck.get("scaler")).__name__,
            "model_present": ck.get("model") is not None,
            "ema_updates": ck.get("updates"), "lr": [x.get("lr") for x in ck["optimizer"]["param_groups"]],
            "train_args_epochs": ck.get("train_args", {}).get("epochs"),
            "train_args_device": ck.get("train_args", {}).get("device"),
            "ultralytics": ck.get("version")}
    del ck
    return info

"""Audit and package the existing E3 epoch-10 Kaggle result, without training."""

import argparse
import csv
import hashlib
import json
import re
import shutil
import tempfile
import zipfile
from pathlib import Path


EXPECTED_SOURCE_SHA = "e60c3f490e25636c1a03fbef6e55bc7add534d760ffced84919b95c307646983"
EXPERIMENT = "E3_pretrained"
WEIGHTS = ("best.pt", "last.pt", "resume_epoch010.pt", "recovery_full.pt")


def _json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def _rows(path):
    with Path(path).open(encoding="utf-8", newline="") as handle:
        rows = [{k.strip(): v.strip() for k, v in row.items()}
                for row in csv.DictReader(handle)]
    if [int(row["epoch"]) for row in rows] != list(range(1, 11)):
        raise ValueError("E3 results.csv must contain exactly epochs 1–10")
    return rows


def _load_for_inference(path, yolo_type):
    model = yolo_type(str(path))
    if model.task != "detect" or len(model.names) != 25:
        raise ValueError(f"Checkpoint is not a 25-class detector: {path}")
    return {"load": "PASS", "task": model.task, "classes": len(model.names)}


def audit_source(source_zip, frozen_yaml, torch_module, yaml_module, yolo_type):
    """Read the original Kaggle ZIP and return a validated temporary extraction."""
    source_zip = Path(source_zip)
    if sha256(source_zip) != EXPECTED_SOURCE_SHA:
        raise ValueError("results11.zip SHA256 differs from the audited source")
    temp = tempfile.TemporaryDirectory(prefix="rod_e3_epoch010_")
    try:
        root = Path(temp.name)
        safe_extract(source_zip, root)
        run = root / "rod_main" / EXPERIMENT
        logs = root / "rod_main" / "logs"
        for rel in ("args.yaml", "results.csv"):
            if not (run / rel).is_file():
                raise FileNotFoundError(run / rel)
        for name in WEIGHTS:
            if not (run / "weights" / name).is_file():
                raise FileNotFoundError(run / "weights" / name)
        expected = yaml_module.safe_load(Path(frozen_yaml).read_text(encoding="utf-8"))
        actual = yaml_module.safe_load((run / "args.yaml").read_text(encoding="utf-8"))
        if expected["model"] != "yolo11n.pt" or expected["pretrained"] is not True:
            raise ValueError("Frozen E3 initialization differs")
        if expected["epochs"] != 30 or actual["epochs"] != 30:
            raise ValueError("Original training schedule must remain recorded as 30")
        if effective_args_mismatches(expected, actual, {"data", "project", "name", "model"}):
            raise ValueError("Effective E3 args differ from frozen YAML")
        if actual["model"] != "yolo11n.pt" or actual["name"] != EXPERIMENT:
            raise ValueError("Actual model/name differs from E3")
        common = {k: v for k, v in expected.items() if k not in {"model", "pretrained", "name"}}
        config_hash = canonical_hash(common)
        source_config = _json(logs / "config_audit.json")
        if source_config["common_sha256"] != config_hash:
            raise ValueError("Kaggle config audit fingerprint differs")
        rows = _rows(run / "results.csv")
        metrics = {key: float(rows[-1][column]) for key, column in {
            "precision": "metrics/precision(B)", "recall": "metrics/recall(B)",
            "map50": "metrics/mAP50(B)", "map50_95": "metrics/mAP50-95(B)"}.items()}
        checkpoint_manifest = _json(logs / "checkpoint_manifest.json")
        resume = run / "weights" / "resume_epoch010.pt"
        info = inspect_checkpoint(resume, 10, torch_module)
        if (checkpoint_manifest["latest_recovery"]["sha256"] != info["sha256"] or
                checkpoint_manifest["latest_recovery"]["epoch"] != 10):
            raise ValueError("Checkpoint manifest differs from full-state checkpoint")
        if sha256(run / "weights" / "recovery_full.pt") != info["sha256"]:
            raise ValueError("Recovery and resume checkpoint hashes differ")
        models = {}
        for name in WEIGHTS:
            path = run / "weights" / name
            raw = torch_module.load(path, map_location="cpu", weights_only=False)
            models[name] = {"sha256": sha256(path), "bytes": path.stat().st_size,
                            "raw_epoch": raw.get("epoch"),
                            "optimizer_present": raw.get("optimizer") is not None,
                            "ema_present": raw.get("ema") is not None,
                            "train_args_epochs": raw.get("train_args", {}).get("epochs")}
            if models[name]["train_args_epochs"] != 30:
                raise ValueError(f"Historical schedule missing from {name}")
            models[name].update(_load_for_inference(path, yolo_type))
            del raw
        for name in ("best.pt", "last.pt"):
            if models[name]["optimizer_present"]:
                raise ValueError(f"Unexpected optimizer in stripped checkpoint {name}")
        if not models["resume_epoch010.pt"]["optimizer_present"]:
            raise ValueError("Resume checkpoint has no optimizer")
        markers = set(re.findall(r"marker_sha256=([0-9a-fA-F]{64})",
                                 (logs / "execution.log").read_text(encoding="utf-8")))
        if len(markers) != 1:
            raise ValueError("Dataset fingerprint missing or ambiguous in execution.log")
        return temp, {"source_archive": {"path": str(source_zip), "sha256": sha256(source_zip),
                                         "bytes": source_zip.stat().st_size, "crc": "PASS"},
                      "experiment": EXPERIMENT, "completed_epochs": 10,
                      "original_schedule_epochs": 30, "current_main_budget_epochs": 10,
                      "dataset_marker_sha256": next(iter(markers)).lower(),
                      "common_config_sha256": config_hash,
                      "frozen_yaml_sha256": sha256(frozen_yaml),
                      "metrics_epoch_10_valid": metrics, "checkpoints": models,
                      "full_state": info, "original_kaggle_error": _json(logs / "errors.json")}
    except Exception:
        temp.cleanup()
        raise


def finalize(source_zip, frozen_yaml, output, torch_module, yaml_module, yolo_type):
    """Create a standalone E3 package; refuses to overwrite an existing artifact."""
    output = Path(output)
    if output.exists():
        raise FileExistsError(output)
    temp, audit = audit_source(source_zip, frozen_yaml, torch_module, yaml_module, yolo_type)
    try:
        root = Path(temp.name)
        package_root = root / "e3_epoch010"
        package_root.mkdir()
        source = root / "rod_main"
        shutil.copytree(source / EXPERIMENT, package_root / EXPERIMENT)
        (package_root / "logs").mkdir()
        for name in ("checkpoint_manifest.json", "config_audit.json", "environment.json",
                     "execution.log", "training_status.json", "errors.json"):
            shutil.copy2(source / "logs" / name, package_root / "logs" / name)
        shutil.copy2(frozen_yaml, package_root / "e3_pretrained_frozen.yaml")
        files = {p.relative_to(package_root).as_posix(): sha256(p)
                 for p in package_root.rglob("*") if p.is_file()}
        audit["files_sha256"] = files
        (package_root / "manifest.json").write_text(
            json.dumps(audit, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
        output.parent.mkdir(parents=True, exist_ok=True)
        partial = output.with_suffix(output.suffix + ".tmp")
        with zipfile.ZipFile(partial, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=1) as z:
            for path in sorted(package_root.rglob("*")):
                if path.is_file():
                    z.write(path, path.relative_to(package_root).as_posix())
        with zipfile.ZipFile(partial) as z:
            if z.testzip() is not None or set(files) | {"manifest.json"} != set(z.namelist()):
                raise ValueError("Standalone E3 artifact ZIP failed CRC/completeness")
            for rel, digest in files.items():
                if hashlib.sha256(z.read(rel)).hexdigest() != digest:
                    raise ValueError(f"Packaged file checksum mismatch: {rel}")
        partial.replace(output)
        sidecar = output.with_suffix(output.suffix + ".sha256")
        sidecar.write_text(f"{sha256(output)}  {output.name}\n", encoding="ascii")
        return {"status": "PASS_LOCAL", "artifact": str(output), "bytes": output.stat().st_size,
                "sha256": sha256(output), "manifest": audit}
    finally:
        temp.cleanup()


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--source-zip", type=Path, required=True)
    parser.add_argument("--frozen-yaml", type=Path, required=True)
    parser.add_argument("--output", type=Path, required=True)
    args = parser.parse_args()
    import torch
    import yaml
    from ultralytics import YOLO
    result = finalize(args.source_zip, args.frozen_yaml, args.output, torch, yaml, YOLO)
    print(json.dumps({k: v for k, v in result.items() if k != "manifest"}, indent=2))



In [ ]:
from pathlib import Path
input_root = Path('/kaggle/input')
matches = [Path(SOURCE_ZIP)] if SOURCE_ZIP else list(input_root.rglob('results11.zip'))
if len(matches) != 1 or not matches[0].is_file():
    raise FileNotFoundError('Attach exactly one results11.zip or set SOURCE_ZIP')
yaml_matches = [Path(FROZEN_YAML)] if FROZEN_YAML else list(input_root.rglob('e3_pretrained.yaml'))
if len(yaml_matches) != 1 or not yaml_matches[0].is_file():
    raise FileNotFoundError('Attach exactly one e3_pretrained.yaml or set FROZEN_YAML')
result = finalize(matches[0], yaml_matches[0], Path(OUTPUT), torch, yaml, YOLO)
print({key: value for key, value in result.items() if key != 'manifest'})


Save a Kaggle Version with outputs, verify the artifact and SHA256 sidecar in the saved Version, then download a backup. An output under `/kaggle/working` alone is not durable storage.
